## 07.2 — Regional specificity of fibroblast subtypes across 41 human brain regions

Wang et al. 2026 (`Data/mmc9.pdf`) resolve human brain fibroblasts into 5 subtypes defined by their **anatomic niche** (dura / arachnoid / pia / parenchyma; see `07.1`), but their own atlas and spatial validation are restricted to the temporal cortex and hippocampus, and the paper explicitly lists as an open question: *"further studies are required to determine how vascular cell states and spatial organization vary across other brain regions."*

Our dataset — 27,185 fibroblast nuclei from **41 brain regions** across **4 donors** (`Data/region_update.csv` gives the region → layer lookup) — is a direct, much broader test of that question. This notebook asks three things:

1. **Composition:** Are Fib_1–Fib_6 (mapped in `07.1` to dural border / large-vessel / inner-arachnoid / arachnoid-barrier / parenchymal-perivascular / barrier-restricted identities) distributed across the 41 regions the way their niche assignment predicts (e.g., barrier-type fibroblasts enriched in choroid plexus + leptomeninges, parenchymal fibroblasts enriched everywhere else)?
2. **Robustness:** Given only 4 donors and very unequal per-region sampling, how much of any regional enrichment is consistent across donors rather than an artifact of pooling cells across donors (pseudoreplication)?
3. **Within-niche regional gradients:** Even after accounting for which niche/subtype is present, does a *given* fibroblast subtype (e.g., parenchymal perivascular, Fib_1) itself carry a further transcriptional signature of which brain region it came from — analogous to the HOX-code positional memory well described in fibroblasts elsewhere in the body?

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(ggplot2)
    library(schard)
    library(viridis)
    library(RColorBrewer)
    library(ComplexHeatmap)
    library(colorRamp2)
    library(edgeR)
    library(patchwork)
})

dir <- "/home/kibr/Work/Fibroblast_regional_difference/"
setwd(dir)
dir.create("./Results/DEG", showWarnings = FALSE, recursive = TRUE)
dir.create("./Results/Tables", showWarnings = FALSE, recursive = TRUE)
dir.create("./Figures", showWarnings = FALSE, recursive = TRUE)

## Note: the teal monochrome palette used for Cell_type in 06.4 (#00695C..#80CBC4) is too
## low-contrast between adjacent Fib_n to read reliably in a stacked bar chart -- use a
## qualitative palette here instead so composition figures are legible.
celltype_colors <- c(
  "Fib_1" = "#1B9E77", "Fib_2" = "#D95F02", "Fib_3" = "#7570B3",
  "Fib_4" = "#E7298A", "Fib_5" = "#66A61E", "Fib_6" = "#E6AB02"
)
region_color <- c(
    "Cortex" = "#009E73", "Brainstem" = "#D55E00", "Cerebellum" = "#046299",
    "Limbic" = "#03B8D8", "Watershed" = "#E69F00", "White Matter Tracts" = "#CC79A7",
    "Olfactory" = "#999999", "Barrier" = "#9C0AE0", "Major Vessel" = "#FF0000"
)

## proposed identities from 07.1 (Results/Tables/Fib_celltype_matching_summary.csv)
match_tbl <- read.csv("./Results/Tables/Fib_celltype_matching_summary.csv")
fib_label <- setNames(paste0(match_tbl$Cell_type, "\n(", match_tbl$Proposed_identity, ")"), match_tbl$Cell_type)
short_label <- setNames(c("Parenchymal perivascular","Large-vessel adventitial","Dural border",
                           "Inner arachnoid","Arachnoid barrier","Barrier-restricted (tentative)"),
                         c("Fib_1","Fib_2","Fib_3","Fib_4","Fib_5","Fib_6"))
short_label

### Load data (same QC as `06.4`/`07.1`) and region lookup table

In [ ]:
h5ad_file <- "./Data/Fibroblast_temp_processed.h5ad"
obj_oi <- schard::h5ad2seurat(h5ad_file)

meta <- obj_oi@meta.data
sample_excluded <- names(table(meta$sampleID))[table(meta$sampleID) < 3]
obj_oi <- subset(obj_oi, subset = !sampleID %in% sample_excluded)
obj_oi <- NormalizeData(obj_oi)
Idents(obj_oi) <- obj_oi$Cell_type

region_lookup <- read.csv("./Data/region_update.csv")
head(region_lookup)

meta <- obj_oi@meta.data
cat("n cells:", nrow(meta), " | n donors:", n_distinct(meta$individualID),
    " | n regions (region_abb):", n_distinct(meta$region_abb),
    " | n region_layer:", n_distinct(meta$region_layer), "\n")

## Part 1 — Compositional specificity: which fibroblast subtype, in which region?

### 1a. region_layer level (9 anatomic layers)

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 6, repr.plot.res = 150)
counts <- meta %>% count(Cell_type, region_layer, name = "freq")
p_bar <- ggplot(counts, aes(fill = Cell_type, y = freq, x = region_layer)) +
    geom_bar(position = "fill", stat = "identity") +
    theme_minimal() + RotatedAxis() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 12),
          axis.text.y = element_text(size = 12)) +
    scale_fill_manual(values = celltype_colors) +
    xlab("") + ylab("Fraction of fibroblast nuclei")
p_bar
ggsave("./Figures/Figure_07_2A_composition_by_region_layer.pdf", p_bar, width = 11, height = 6)

### 1b. Pooled-cell enrichment test (region_layer × Cell_type)

A chi-squared test on all 27,182 cells pooled across donors — same approach as `06.4`. This is fast and visual but **pseudoreplicated** (donors contribute very unevenly, e.g. one donor supplies 92% of Barrier cells), so p-values here are optimistic; treat this panel as descriptive effect size (log observed/expected ratio) and cross-check with the donor-aware analysis in 1c below.

In [ ]:
obs_mat <- as.matrix(table(meta$region_layer, meta$Cell_type))
chisq_res <- chisq.test(obs_mat)
exp_mat <- chisq_res$expected
ratio <- log((obs_mat + 1) / (exp_mat + 1))  # pseudocount to avoid -Inf on zero cells
colnames(ratio) <- short_label[colnames(ratio)]
tratio <- t(ratio)

make_ht_2B <- function() {
    Heatmap(tratio, name = "log2 enrichment",
            col = colorRamp2(c(-2, 0, 2), c("#5387be", "#F7F7F7", "#b31f2c")),
            cluster_rows = FALSE, cluster_columns = FALSE,
            bottom_annotation = HeatmapAnnotation(region = colnames(tratio),
                                                   col = list(region = region_color), show_legend = FALSE),
            row_names_side = "left", column_names_rot = 40,
            cell_fun = function(j, i, x, y, w, h, fill) grid.text(sprintf("%.1f", tratio[i,j]), x, y, gp = gpar(fontsize = 8)))
}
pdf("./Figures/Figure_07_2B_region_layer_enrichment_heatmap.pdf", width = 7, height = 4.5)
draw(make_ht_2B())
dev.off()
draw(make_ht_2B())
cat("chi-sq (pooled cells, all df) p-value:", format.pval(chisq_res$p.value), " -- inflated by pseudoreplication, see 1c\n")

### 1c. Donor-aware robustness check

For each Cell_type × region_layer pair (restricted to donor×region_layer combinations with ≥10 total fibroblast nuclei so per-donor proportions are not too noisy), we compute each donor's own proportion of that subtype among all fibroblasts sampled from that region_layer, then ask: in how many of the informative donors is the subtype's local proportion *above* its global average (i.e., enriched)? This does not pool cells across donors, so it directly addresses pseudoreplication.

In [ ]:
comp <- meta %>% count(individualID, region_layer, Cell_type, name = "n", .drop = FALSE)
tot <- meta %>% count(individualID, region_layer, name = "total", .drop = FALSE)
comp <- merge(comp, tot, by = c("individualID", "region_layer"))
comp$prop <- comp$n / comp$total
comp <- comp %>% filter(total >= 10)

global_prop <- meta %>% count(Cell_type, name = "n") %>% mutate(global_p = n / sum(n)) %>% dplyr::select(Cell_type, global_p)

consist <- comp %>%
    left_join(global_prop, by = "Cell_type") %>%
    mutate(enriched = prop > global_p) %>%
    group_by(Cell_type, region_layer) %>%
    summarise(n_donors = n(), n_enriched = sum(enriched), mean_prop = mean(prop), .groups = "drop") %>%
    mutate(consistent_enrichment = n_enriched == n_donors,
           consistent_depletion  = n_enriched == 0) %>%
    arrange(Cell_type, desc(mean_prop))

write.csv(consist, "./Results/Tables/regional_composition_donor_consistency.csv", row.names = FALSE)
print(consist, n = 60)

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 7, repr.plot.res = 150)
comp_annot <- comp %>% left_join(global_prop, by = "Cell_type") %>%
    mutate(Cell_type_label = short_label[Cell_type])
p_dots <- ggplot(comp_annot, aes(x = region_layer, y = prop, color = individualID)) +
    geom_hline(aes(yintercept = global_p), linetype = "dashed", color = "grey50") +
    geom_point(size = 2, position = position_jitter(width = 0.15, height = 0)) +
    facet_wrap(~Cell_type_label, scales = "free_y", ncol = 3) +
    theme_bw() + RotatedAxis() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 8)) +
    ylab("Proportion of fibroblasts in that donor x region_layer") + xlab("") +
    labs(color = "Donor", caption = "Dashed line = global mean proportion for that subtype. Only donor x region_layer combos with >=10 fibroblast nuclei shown.")
p_dots
ggsave("./Figures/Figure_07_2C_donor_level_composition.pdf", p_dots, width = 11, height = 7)

### 1d. Fine-grained region_abb level (41 regions)

In [ ]:
obs_mat2 <- as.matrix(table(meta$region_abb, meta$Cell_type))
chisq_res2 <- chisq.test(obs_mat2)
ratio2 <- log((obs_mat2 + 1) / (chisq_res2$expected + 1))
colnames(ratio2) <- short_label[colnames(ratio2)]

## note: in region_update.csv, Region_layer_1 is the full descriptive region name and
## Region_layer_2 is the coarse anatomic layer that matches obj_oi$region_layer
region_layer_of <- region_lookup$Region_layer_2[match(rownames(ratio2), region_lookup$Final_abb)]

## order rows by region_layer then by row name for readability
ord <- order(factor(region_layer_of, levels = names(region_color)), rownames(ratio2))

make_ht_2D <- function() {
    row_ha <- rowAnnotation(region_layer = region_layer_of[ord], col = list(region_layer = region_color),
                             annotation_name_side = "top")
    Heatmap(ratio2[ord, ], name = "log2 enrichment",
            col = colorRamp2(c(-3, 0, 3), c("#5387be", "#F7F7F7", "#b31f2c")),
            cluster_rows = FALSE, cluster_columns = FALSE,
            right_annotation = row_ha,
            row_names_side = "left", column_names_rot = 90, row_names_gp = gpar(fontsize = 8))
}
pdf("./Figures/Figure_07_2D_region_abb_enrichment_heatmap.pdf", width = 8, height = 11)
draw(make_ht_2D())
dev.off()
draw(make_ht_2D())

## Part 2 — Within-niche regional gradients: does a single fibroblast subtype vary by brain region?

Composition tells us *which* subtype dominates in a region; it does not tell us whether a given subtype itself carries further region-specific transcriptional identity. We test this within **Fib_1 (parenchymal perivascular FB)**, the subtype broadly sampled across the neuraxis with the best donor×region coverage, using `Broad Embryologic Lineage` (Telencephalon → Diencephalon → Mesencephalon → Rhombencephalon → Spinal cord, i.e. rostral → caudal) as the grouping variable and pseudobulk edgeR (donor as covariate), mirroring the design used for the Cell_type-level DEG analysis in `06.4`.

In [ ]:
fib1 <- subset(obj_oi, subset = Cell_type == "Fib_1")
fib1$lineage <- fib1$`Broad Embryologic Lineage`
keep_lineage <- c("Telencephalon", "Telencephalon-associated tract/white matter",
                   "Diencephalon", "Mesencephalon", "Rhombencephalon", "Spinal cord")
fib1 <- subset(fib1, subset = lineage %in% keep_lineage)
fib1$lineage <- droplevels(factor(fib1$lineage, levels = keep_lineage))

table(fib1$lineage, fib1$individualID)

In [ ]:
pseudo <- AggregateExpression(fib1, assays = "RNA", return.seurat = TRUE, group.by = "sampleID")
mat <- pseudo[["RNA"]]$counts

smeta <- fib1@meta.data %>% distinct(sampleID, individualID, lineage, region_layer, region_abb)
smeta$key <- gsub("_", "-", as.character(smeta$sampleID))
rownames(smeta) <- smeta$key
smeta <- smeta[colnames(mat), ]

y <- DGEList(counts = mat, group = smeta$lineage)
y$samples$donor <- smeta$individualID
keep.samples <- y$samples$lib.size > 2000
y <- y[, keep.samples]
smeta <- smeta[keep.samples, ]
mat_kept <- mat[, keep.samples]  # raw (gene-unfiltered) counts for kept samples, used later for the HOX/AP scan

group <- droplevels(factor(smeta$lineage))
donor <- droplevels(factor(smeta$individualID))
table(group)

design <- model.matrix(~0 + group + donor)
colnames(design) <- make.names(colnames(design))
keep <- filterByExpr(y, design = design)
y <- y[keep, , keep.lib.sizes = FALSE]
y <- normLibSizes(y)
y <- estimateDisp(y, design, robust = TRUE)
fit <- glmQLFit(y, design, robust = TRUE)
cat(sum(keep), "genes tested across", ncol(y), "pseudobulk samples from", nlevels(donor), "donors and", nlevels(group), "lineage groups\n")

**Global (ANOVA-like) test:** is there *any* lineage effect on Fib_1 pseudobulk expression, beyond donor?

In [ ]:
grp_cols <- grep("^group", colnames(design), value = TRUE)
Cmat <- matrix(0, nrow = ncol(design), ncol = length(grp_cols) - 1, dimnames = list(colnames(design), NULL))
for (i in 2:length(grp_cols)) { Cmat[grp_cols[1], i - 1] <- 1; Cmat[grp_cols[i], i - 1] <- -1 }
colnames(Cmat) <- paste0(grp_cols[-1], "_vs_", grp_cols[1])

qlf <- glmQLFTest(fit, contrast = Cmat)
tt <- topTags(qlf, n = Inf)$table
n_sig <- sum(tt$FDR < 0.05)
cat(n_sig, "of", nrow(tt), "tested genes are lineage-associated at FDR<0.05 within Fib_1\n")
write.csv(tt, "./Results/DEG/Fib1_lineage_edgeR_global_test.csv")
head(tt[order(tt$FDR), ], 20)

**Caveat:** the top hits are dominated by heat-shock/stress genes (`HSP90AA1`, `HSPA1A`, `HSPA1B`, `DNAJB1`), a classic dissociation/tissue-handling signature that often correlates with which regions were processed in which batch — this is very likely a technical confound rather than biology, and should not be over-interpreted region by region. Genes further down the ranked list (`TSHZ2`, `NKD1`, `GPC3`, `CACNA2D3`, `DLG2`, `SLC4A7`, `CASC15`) are candidate genuine regional-identity genes worth independent validation; several are notable because they sit in Wnt-signaling (`NKD1`) or are used elsewhere in this very atlas as an arteriovenous zonation marker (`TSHZ2`, a vein marker in Wang et al.'s endothelial analysis), suggesting fibroblasts may echo positional signaling gradients rather than a distinct fibroblast-autonomous code.

### Positional-identity ("HOX-code") check

Fibroblasts throughout the rest of the body retain a developmental HOX-code that marks their body-axis position long after development (e.g., Rinn et al. 2006; Muhl et al. 2020 fibroblast atlas — cited in Wang et al.'s own reference list). We test the analogous idea along the brain's rostrocaudal (neuraxis) axis: do HOX genes and upstream retinoic-acid/AP-patterning genes show any residual expression gradient in Fib_1 across `Broad Embryologic Lineage`, from telencephalon (rostral) to spinal cord (caudal)?

In [ ]:
## use the raw (gene-unfiltered) pseudobulk counts here -- filterByExpr above would have
## dropped most HOX genes since they are lowly/rarely expressed in fibroblasts
cpm_mat <- edgeR::cpm(mat_kept, log = TRUE, prior.count = 1)
hox_genes <- grep("^HOX[A-D][0-9]", rownames(cpm_mat), value = TRUE)
ap_genes <- intersect(c("EN1","EN2","OTX2","GBX2","FOXG1","LHX9","SIX3","EMX2","ALDH1A2","RSPO2","RSPO3","BARHL1","TBX2","WNT7B"),
                       rownames(cpm_mat))
genes_use <- c(sort(hox_genes), ap_genes)

sub_mat <- cpm_mat[genes_use, , drop = FALSE]
mean_by_lineage <- sapply(split(seq_len(ncol(sub_mat)), group), function(idx) rowMeans(sub_mat[, idx, drop = FALSE]))
mean_by_lineage <- mean_by_lineage[, intersect(keep_lineage, colnames(mean_by_lineage))]
write.csv(mean_by_lineage, "./Results/Tables/Fib1_HOX_AP_gene_meanCPM_by_lineage.csv")

options(repr.plot.width = 7, repr.plot.height = 9, repr.plot.res = 150)
make_ht_2E <- function() {
    Heatmap(mean_by_lineage, name = "mean logCPM",
            col = colorRamp2(c(min(mean_by_lineage), median(mean_by_lineage), max(mean_by_lineage)),
                              c("#F7F7F7", "#f4a582", "#b2182b")),
            cluster_rows = FALSE, cluster_columns = FALSE,
            row_names_gp = gpar(fontsize = 8), column_names_rot = 90,
            column_title = "Fib_1 pseudobulk mean expression, rostral (left) to caudal (right)")
}
pdf("./Figures/Figure_07_2E_HOX_AP_gene_heatmap.pdf", width = 7, height = 9.5)
draw(make_ht_2E())
dev.off()
draw(make_ht_2E())

**Result.** Nearly all HOX genes and forebrain/midbrain AP-TFs sit at floor expression in fibroblasts regardless of region (consistent with these being neuronal/progenitor patterning genes, not normally active in adult mesenchyme). Two genes are the clear exceptions: **`HOXB3`** and **`HOXD3`** (both paralog-group-3 HOX genes, whose expression boundary in development sits at the hindbrain/spinal-cord junction) are selectively elevated in the **spinal cord** samples only -- and only there, consistent with a real positional-identity echo rather than a general caudal trend. **`ALDH1A2`** (retinoic-acid synthesis, the canonical upstream driver of posterior HOX activation) is high across essentially every lineage including telencephalon, i.e. it behaves as a general leptomeningeal/perivascular-fibroblast identity marker in this dataset rather than an AP-graded one -- worth noting since RA signaling is the textbook driver of posterior HOX activation, but here it does not track rostrocaudal position. This is a small, likely underpowered signal (Diencephalon/Mesencephalon/Spinal-cord lineage groups each rely on only 1 sample per donor in ≤3 donors — see caveats below), but the HOXB3/HOXD3 spinal-cord-restriction is directionally exactly what a residual HOX-code echo would look like, and is consistent with the general finding in the fibroblast literature (mouse and human, non-CNS) that fibroblasts retain positional transcriptional memory of their embryonic origin.

## Summary and interpretation

**1. Composition dominates.** The strongest, most consistent regional signal in this dataset is *which fibroblast subtype is present*, not how a given subtype's transcriptome varies. Barrier-type niches (choroid plexus + leptomeninges) are essentially the exclusive home of Fib_4 (inner arachnoid), Fib_5 (arachnoid barrier) and Fib_6; the three named large cerebral arteries (MCA/ACA/BA) are almost exclusively Fib_2 (large-vessel adventitial); and every sampled parenchymal region (cortex, limbic structures, brainstem, cerebellum, white matter tracts, watershed zones) is dominated by Fib_1 (parenchymal perivascular). This mirrors — and, for a much larger set of regions than the original cortex/hippocampus study, directly extends — the niche-based organizing principle Wang et al. proposed for human brain fibroblasts.

**2. The niche principle generalizes beyond cortex/hippocampus, with one clear addition.** All 41 regions, including the olfactory bulb and every neuraxis level down to spinal cord, resolve into the same handful of niche-defined subtypes rather than acquiring wholesale new fibroblast states — supporting the idea that fibroblast identity in the human brain is fundamentally set by local anatomic niche (dura/arachnoid/pia/parenchyma), not by gross brain region per se. The one clear departure is Fib_2: it is essentially absent from the cortex/hippocampus/choroid-plexus/leptomeninges compartments that the paper sampled and instead dominates the three named large arteries, which the reference atlas never captured directly — this looks like a caliber-scaled extension of the pial-fibroblast identity rather than evidence against the niche framework.

**3. A secondary, subtler regional axis exists within a niche.** Even holding subtype constant (Fib_1 only), pseudobulk expression shows a modest but real region-associated signal (49/3899 tested genes at FDR<0.05), and a HOX-code-like echo (HOXB3 and HOXD3, both selectively restricted to the spinal cord) appears specifically at the caudal-most end of the sampled neuraxis. This is a much weaker determinant of fibroblast state than niche identity, is confounded in part by technical (dissociation-stress) signal, and is based on limited donor replication for the rarer lineage groups — but it raises a concrete, testable hypothesis for follow-up work: human brain fibroblasts may carry a faint, body-plan-consistent positional memory layered on top of their dominant niche identity, analogous to fibroblasts elsewhere in the body.

**Key caveats:** (i) only 4 donors, with very unequal per-region and per-lineage sampling (e.g., 2 donors have no Major Vessel coverage after QC, several rare lineage groups rely on 1 sample/donor); (ii) pooled-cell chi-squared tests (1b, 1d) are pseudoreplicated and should be read as effect sizes, cross-checked against the donor-aware analysis (1c); (iii) Fib_6 in particular is small (254 cells, 80% from one donor) and partly marked by dissociation-stress genes, so its "Barrier-restricted" status is provisional.